# Quadratic elements and native Gmsh formats

`mesh_order=2` produces 6-node triangles (Gmsh type 9). Connectivity is stored
with 6 columns; `plot_by_grain` fills the first 3 corners.

**Abaqus INP** writes **CPS6/CPE6** (6-node tri) or **CPS8/CPE8** (8-node quad)
when connectivity has mid-side nodes. Gmsh node order matches Abaqus (corners
then mids). Native `.msh`/`.vtk` are still written via `formats=` while the
Gmsh session is live.

Requires `gmsh` (`pip install upxo[mesh]`). `confMesh2d` (pygmsh) is deprecated.
Canonical mesh-only demo: `confMesh2d_gmsh.ipynb`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import box

from upxo.meshing.gsmesh2d import mesh_gs
from upxo.meshing.writer_ABQ import summarize_inp


In [ ]:
cells = {
    1: box(0, 0, 2, 2),
    2: box(2, 0, 4, 2),
    3: box(0, 2, 2, 4),
    4: box(2, 2, 4, 4),
}

In [ ]:
out = Path.cwd() / 'confMesh2d_quadratic_out'
lin = mesh_gs(cells, mesh_size_gb=0.4, mesh_size_bulk=0.8,
              mesh_order=1, mesh_algo=6, recombine_to_quads=False)
q2 = mesh_gs(cells, mesh_size_gb=0.4, mesh_size_bulk=0.8,
             mesh_order=2, mesh_algo=6, recombine_to_quads=False,
             out_dir=str(out), formats=['msh', 'vtk'], basename='rve_q2')
mq = q2['mesher']
mq.form_elsets_gmsh(); mq.build_boundary_nsets(); mq.build_gb_nset()
print('linear nodes', lin['n_nodes'], 'quadratic nodes', q2['n_nodes'])
print('quadratic conn', mq.elConn['triangle'].shape, 'order', mq.elementOrder)
print('native files', q2['exported'])
print(mq.validation_report)

In [ ]:
fig, ax = mq.plot_by_grain(figsize=(6, 6), show_gb=True, show_nsets=True,
                           title='quadratic tris (corners plotted)')
fig

In [ ]:
inp_s = mq.export_abaqus_inp(out / 'rve_cps6.inp', plane='stress')
inp_e = mq.export_abaqus_inp(out / 'rve_cpe6.inp', plane='strain')
text = Path(inp_s).read_text(encoding='utf-8')
assert '*Element, type=CPS6' in text
inp_s, summarize_inp(inp_s), inp_e, q2['exported']